# 04 (v2) - Train YOLO: board-level split, folds, seeds, Drive mirroring + resume
Runtime: **GPU**. If the session dies, re-run cells 1-3 and training resumes from Drive.

Fixes vs v1: val/test no longer point at the same dir; split dirs are built from the v2 manifest; synthetic data is added **only from train boards** of the chosen split; training cell no longer kills the kernel and the final cell is gated on `best.pt`; results go to `master_results_v2.csv` (fold, seed, synth ratio) with a mean +- std summary.

**Experiment grid** (change the 3 variables in cell 2, run, repeat):
`USE_SYNTH` in {False, True} x `FOLD` in {0,1,2,3} (or `None` = fixed split) x `SEED` in {42, 43, ...}.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import sys, os
D = '/content/drive/MyDrive/pcbrisk'
sys.path.insert(0, f'{D}/code')
from pcbrisk_utils import *
seed_all(42)

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
sh('pip -q install ultralytics onnx onnxruntime')

Mounted at /content/drive
GPU: Tesla T4


CompletedProcess(args='pip -q install ultralytics onnx onnxruntime', returncode=0)

In [2]:
USE_SYNTH   = False
SYNTH_RATIO = 1.0        # synthetic train images = ratio x real train images
FOLD        = None       # None = fixed 6/2/3 split; 0..3 = leave-boards-out fold
SEED        = 42
EPOCHS, IMGSZ, BATCH = 60, 640, 16

fold_tag = 'fixed' if FOLD is None else f'f{FOLD}'
syn_tag = f'+synth{SYNTH_RATIO:g}' if USE_SYNTH else ''
exp_id = f'det_yolo11n_deeppcb{syn_tag}_{fold_tag}_s{SEED}_v2'
print(exp_id)

det_yolo11n_deeppcb_fixed_s42_v2


In [3]:
import pandas as pd, numpy as np, shutil, yaml
from pathlib import Path

restore_tar(f'{D}/01_prepared/prepared_v2.tar', f'{L}/data', marker='.ready_prepared_v2')
mp = f'{D}/01_prepared/split_manifest_v2.csv'
assert sha256_file(mp) == open(f'{D}/01_prepared/manifest_v2.sha256').read(), 'manifest changed'
man = pd.read_csv(mp, dtype={'image_id': str})

m = man.copy()
if FOLD is not None:
    m['split'] = np.where(m.fold == FOLD, 'test', np.where(m.fold == (FOLD + 1) % 4, 'val', 'train'))

S = Path(f'{L}/data/split'); shutil.rmtree(S, ignore_errors=True)
for s in ['train', 'val', 'test']:
    (S/'images'/s).mkdir(parents=True); (S/'labels'/s).mkdir(parents=True)
Y = Path(f'{L}/data/yolo_all')
for r in m.itertuples():
    shutil.copy2(Y/'images'/f'{r.image_id}.jpg', S/'images'/r.split/f'{r.image_id}.jpg')
    shutil.copy2(Y/'labels'/f'{r.image_id}.txt', S/'labels'/r.split/f'{r.image_id}.txt')

# board-level leakage check
b = m.groupby('split').source_board.agg(set)
assert not (b['train'] & b['val']) and not (b['train'] & b['test']) and not (b['val'] & b['test']), 'board leakage'
train_boards = b['train']

if USE_SYNTH:
    restore_tar(f'{D}/02_synth/synth_v2.tar', f'{L}/data', marker='.ready_synth_v2')
    sm = pd.read_csv(f'{L}/data/synth/synth_meta.csv', dtype={'template': str})
    sm = sm[sm.source_board.isin(train_boards)]                  # synthetic only from TRAIN boards
    n = int(SYNTH_RATIO * (m.split == 'train').sum())
    sm = sm.sample(min(n, len(sm)), random_state=SEED)
    for sid in sm.synth_id:
        shutil.copy2(f'{L}/data/synth/images/{sid}.jpg', S/'images'/'train'/f'{sid}.jpg')
        shutil.copy2(f'{L}/data/synth/labels/{sid}.txt', S/'labels'/'train'/f'{sid}.txt')
    assert set(sm.source_board) <= train_boards
    print(len(sm), 'synthetic images added to train')

print({s: len(list((S/'images'/s).glob('*.jpg'))) for s in ['train', 'val', 'test']})
cc = [f'c{i}' for i in range(6)]
print(m.groupby('split')[cc].sum())

NAMES = ['open', 'short', 'mousebite', 'spur', 'copper', 'pin-hole']
yaml.safe_dump(dict(path=str(S), train='images/train', val='images/val', test='images/test',
                    names={i: n for i, n in enumerate(NAMES)}), open(f'{L}/data/data.yaml', 'w'))

{'train': 882, 'val': 218, 'test': 400}
         c0   c1    c2   c3   c4   c5
split                                
test    549  402   591  456  401  422
train  1118  929  1159  966  889  871
val     275  175   215  203  184  208


In [4]:
from ultralytics import YOLO
import time
RD = f'{D}/04_runs/{exp_id}'; os.makedirs(f'{RD}/weights', exist_ok=True)
LOCAL = f'{L}/runs/{exp_id}'

last = {'t': 0}
def mirror(trainer):
    now = time.time()
    if now - last['t'] > 120:                                   # throttle Drive writes
        for f in ['weights/last.pt', 'weights/best.pt', 'results.csv', 'args.yaml']:
            p = os.path.join(trainer.save_dir, f)
            if os.path.exists(p): safe_copy(p, f'{RD}/{f}', verify=False)
        last['t'] = now

if os.path.exists(f'{RD}/DONE'):
    print('DONE marker present - skipping training')
else:
    src, dst = f'{RD}/weights/last.pt', f'{LOCAL}/weights/last.pt'
    if os.path.exists(src) and not os.path.exists(dst):
        safe_copy(src, dst, verify=False)
    if os.path.exists(dst):
        mdl = YOLO(dst); mdl.add_callback('on_model_save', mirror); mdl.train(resume=True)
    else:
        mdl = YOLO('yolo11n.pt'); mdl.add_callback('on_model_save', mirror)
        mdl.train(data=f'{L}/data/data.yaml', epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=30,
                  seed=SEED, deterministic=True, workers=min(4, os.cpu_count()),
                  project=f'{L}/runs', name=exp_id, exist_ok=True, cache='disk')

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=disk, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/data/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=N

In [5]:
# final: runs only if training produced best.pt; writes ledger row once
import json, subprocess
best_local = f'{LOCAL}/weights/best.pt'
if not os.path.exists(best_local):
    if os.path.exists(f'{RD}/weights/best.pt'):
        safe_copy(f'{RD}/weights/best.pt', best_local)
    else:
        raise RuntimeError('no best.pt - training cell did not finish; re-run it (it resumes)')

if os.path.exists(f'{RD}/DONE'):
    print('already recorded')
else:
    for f in ['weights/last.pt', 'weights/best.pt', 'results.csv', 'args.yaml']:
        p = f'{LOCAL}/{f}'
        if os.path.exists(p): safe_copy(p, f'{RD}/{f}')
    info = dict(exp_id=exp_id, seed=SEED, fold=FOLD, use_synth=USE_SYNTH, synth_ratio=SYNTH_RATIO,
                epochs=EPOCHS, imgsz=IMGSZ, torch=torch.__version__,
                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
                split_hash=open(f'{D}/01_prepared/manifest_v2.sha256').read(),
                freeze=subprocess.run('pip freeze', shell=True, capture_output=True, text=True).stdout.splitlines())
    json.dump(info, open(f'{RD}/run_info.json', 'w'), indent=1)

    best = YOLO(best_local)
    r = best.val(data=f'{L}/data/data.yaml', split='test', imgsz=IMGSZ)
    row = pd.DataFrame([dict(exp_id=exp_id, use_synth=USE_SYNTH, synth_ratio=SYNTH_RATIO if USE_SYNTH else 0,
                             fold=fold_tag, seed=SEED, test_map50=r.box.map50, test_map50_95=r.box.map)])
    led = f'{D}/05_results/master_results_v2.csv'
    row.to_csv(led, mode='a', header=not os.path.exists(led), index=False)
    open(f'{RD}/DONE', 'w').write('ok')
    print(row.to_string(index=False))

Ultralytics 8.4.174 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 100 layers, 2,583,322 parameters, 0 gradients, 6.4 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 15.7±13.0 MB/s, size: 42.1 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/data/split/labels/test... 400 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 400/400 1.0Kit/s 0.4s
val: New cache created: /content/data/split/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 25/25 4.3it/s 5.9s
                   all        400       2821      0.943      0.913      0.963      0.695
                  open        391        549      0.877      0.936      0.936      0.623
                 short        320        402      0.939      0.886      0.967      0.619
             m

In [6]:
# summary across all finished runs (needs several folds/seeds to mean anything)
led = pd.read_csv(f'{D}/05_results/master_results_v2.csv')
print(led.groupby(['use_synth', 'synth_ratio']).test_map50.agg(['mean', 'std', 'count']).round(4))
print(led.pivot_table(index='fold', columns='use_synth', values='test_map50', aggfunc='mean').round(4))

                         mean  std  count
use_synth synth_ratio                    
False     0            0.9625  NaN      1
use_synth   False
fold             
fixed      0.9625


## Optional: ONNX export + parity (run only for the final chosen model)

In [8]:
import glob
RUN_ONNX = False
if RUN_ONNX:
    best = YOLO(best_local)
    onnx_path = best.export(format='onnx', imgsz=IMGSZ, dynamic=False)
    safe_copy(onnx_path, f'{D}/06_exports/{exp_id}.onnx')
    imgs = sorted(glob.glob(f'{S}/images/test/*.jpg'))[:100]
    pt = YOLO(best_local); ox = YOLO(onnx_path, task='detect')
    same_n = same_cls = 0
    for p in imgs:
        a = pt.predict(p, conf=0.25, verbose=False)[0].boxes; bx = ox.predict(p, conf=0.25, verbose=False)[0].boxes
        if len(a) == len(bx):
            same_n += 1
            same_cls += int(np.array_equal(np.sort(a.cls.cpu().numpy()), np.sort(bx.cls.cpu().numpy())))
    print(f'same #detections {same_n}/{len(imgs)} | same classes {same_cls}/{len(imgs)}')